# 🚀 Модуль 2. Урок 5. Оконные функции (Window Functions): ранжирование, кумулятивные итоги и смещения

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_05_window_functions_advanced.ipynb)

## 📌 Бизнес-контекст и методология
Оконные функции вычисляют агрегаты по группе строк, не схлопывая исходные записи.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(ensure_dataset("ecommerce.db"))

query = """
WITH OrderTotals AS (
    SELECT 
        order_id,
        SUM(total_amount) AS order_total
    FROM order_items
    GROUP BY order_id
)
SELECT 
    o.customer_id,
    o.order_id,
    o.order_date,
    ot.order_total,
    ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date) AS order_seq,
    ROUND(SUM(ot.order_total) OVER (PARTITION BY o.customer_id ORDER BY o.order_date), 2) AS running_ltv,
    LAG(ot.order_total, 1) OVER (PARTITION BY o.customer_id ORDER BY o.order_date) AS prev_order_total
FROM orders o
JOIN OrderTotals ot ON o.order_id = ot.order_id
WHERE o.status = 'Completed'
ORDER BY o.customer_id, o.order_date
LIMIT 10;
"""

df_window = pd.read_sql_query(query, conn)
df_window


,customer_id,order_id,order_date,order_total,order_seq,running_ltv,prev_order_total
0,1,10001,2026-01-28 00:00:00,1123.61,1,1123.61,NaN
1,1,10002,2026-02-17 00:00:00,1236.36,2,2359.97,1123.61
2,1,10003,2026-03-15 00:00:00,1551.84,3,3911.81,1236.36
3,2,10004,2025-06-03 00:00:00,843.33,1,843.33,NaN
4,4,10005,2025-05-29 00:00:00,276.22,1,276.22,NaN
5,4,10006,2025-06-21 00:00:00,3409.93,2,3686.15,276.22
6,4,10008,2025-09-03 00:00:00,887.72,3,4573.87,3409.93
7,4,10009,2025-09-22 00:00:00,4375.49,4,8949.36,887.72
8,5,10010,2025-05-06 00:00:00,1072.23,1,1072.23,NaN
9,6,10011,2026-01-15 00:00:00,1583.36,1,1583.36,NaN


### 🎯 Практическое задание 1: Поиск второго чека клиента через DENSE_RANK
Найдите второй по величине чек клиента с помощью `DENSE_RANK()`.

In [3]:
### ВАШ КОД ЗДЕСЬ
query_rank = """
WITH OrderTotals AS (
    SELECT order_id, SUM(total_amount) AS total FROM order_items GROUP BY order_id
),
RankedOrders AS (
    SELECT 
        o.customer_id,
        o.order_id,
        ot.total,
        DENSE_RANK() OVER (PARTITION BY o.customer_id ORDER BY ot.total DESC) AS check_rank
    FROM orders o
    JOIN OrderTotals ot ON o.order_id = ot.order_id
    WHERE o.status = 'Completed'
)
SELECT 
    customer_id,
    order_id,
    total AS second_largest_check
FROM RankedOrders
WHERE check_rank = 2
ORDER BY total DESC;
"""

df_second = pd.read_sql_query(query_rank, conn)
print(df_second.head(5))

assert not df_second.empty, "Должны быть вторые чеки"
assert 'second_largest_check' in df_second.columns, "Колонка обязательна"
print("✅ Тест успешно пройден!")


   customer_id  order_id  second_largest_check
0          517     10979               4068.12
1           85     10174               3938.00
2           24     10062               3490.78
3            4     10006               3409.93
4          698     11324               3149.18
✅ Тест успешно пройден!


### 💼 ROW_NUMBER vs RANK vs DENSE_RANK
ROW_NUMBER всегда дает уникальный номер строки, DENSE_RANK дает одинаковый ранг дубликатам без пропуска следующих номеров.